# Tree circuit: supervision density as the scaling-law dial

One random fan-in-3 tree over 2187 inputs (1093 interior wires, depths 1..7;
every wire exactly balanced). Each arm taps interior wires i.i.d. with
probability **q** and trains only on the tapped set. A tapped node's
effective difficulty is exponential in its **supervision gap** — the depth of
unsupervised computation beneath it — and gaps are percolation-distributed,
so q controls the difficulty distribution *naturally*:

- **q = 1**: full process supervision — staircase cascade, expect saturation.
- **intermediate q**: geometric gap distribution — power-law-ish cost stats.
- **q -> 0**: isolated outcomes — grokking cliffs, deep taps out of reach.

Sized so the large model (~40M params) can saturate everything reachable:
scaffolded gaps stay within the measured ~40M frontier (gap ~4-5 solvable);
rare deep-isolated taps at low q are the designed exception, not the bulk.
Predictions to check: (1) accuracy organizes by *gap*, not depth, uniformly
across q; (2) the q=1 arm saturates; (3) tapped-loss L(D) morphs from
exponential-ish decay (q=1) toward step-like (q=0.03).

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from train import RunConfig, load_run, run
from tree_circuit import sample_tree_circuit, supervision_gaps

N_LEAVES, TREE_DEPTH = 2187, 7
Q_GRID = [1.0, 0.3, 0.1, 0.03]
SHAPES = [(256, 6), (700, 10)]      # ~3.5M / ~40M params
LR = 1e-3                            # tuned value for both shapes (hard task)
STEPS = 50_000
TAP_SEED = 1000
OUT_DIR = "runs/tree"

circuit = sample_tree_circuit(np.random.default_rng(0), N_LEAVES, TREE_DEPTH)
node_depths = circuit.out_depths

taps = {}
for q in Q_GRID:
    if q >= 1.0:
        taps[q] = np.ones(circuit.n_interior, dtype=bool)
    else:
        rng = np.random.default_rng([TAP_SEED, int(q * 1000)])
        taps[q] = rng.random(circuit.n_interior) < q

configs = {}
for w, d in SHAPES:
    for q in Q_GRID:
        ow = None if q >= 1.0 else tuple(np.flatnonzero(taps[q]).tolist())
        configs[(w, d, q)] = RunConfig(
            task="tree3", n_wires=N_LEAVES, circ_depth=TREE_DEPTH,
            width=w, mlp_depth=d, lr=LR, steps=STEPS,
            output_wires=ow, out_dir=OUT_DIR,
        )

print("tapped nodes by depth (columns: depth 1..7):")
for q in Q_GRID:
    counts = [int((taps[q] & (node_depths == k)).sum()) for k in range(1, 8)]
    gaps = supervision_gaps(circuit, taps[q])
    gmax = gaps[taps[q]].max() if taps[q].any() else 0
    print(f"  q={q:<5} n={taps[q].sum():3d}  {counts}  max gap {gmax}")

## Run (~2 h on a T4 at defaults; resumable, skips completed)

In [ ]:
for cfg in configs.values():
    run(cfg)

## Tapped-only loss curves

Loss averaged over each arm's *tapped* outputs (untapped head columns are
untrained noise — always mask them out).

In [ ]:
CHANCE = np.log(2)
done = {k: load_run(c.npz_path)[1] for k, c in configs.items()
        if c.npz_path.exists()}

fig, axes = plt.subplots(1, len(SHAPES), figsize=(5.6 * len(SHAPES), 4.2),
                         sharey=True)
qcols = plt.cm.plasma(np.linspace(0.05, 0.8, len(Q_GRID)))
for ax, (w, dpt) in zip(np.atleast_1d(axes), SHAPES):
    for q, col in zip(Q_GRID, qcols):
        if (w, dpt, q) not in done:
            continue
        d = done[(w, dpt, q)]
        sel = d["eval_steps"] >= 1000
        tw = np.flatnonzero(taps[q])
        ax.plot(d["eval_steps"][sel] * 256,
                d["per_out_loss"][sel][:, tw].mean(axis=1),
                color=col, lw=1.4, label=f"q={q}")
    ax.axhline(CHANCE, color="gray", ls=":", lw=1)
    ax.set(xscale="log", yscale="log", xlabel="samples D",
           title=f"w{w}d{dpt}")
np.atleast_1d(axes)[0].set_ylabel("eval BCE (tapped outputs)")
np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout()

## The money plot: accuracy organizes by gap, not depth

If the scaffolding theory is right, final accuracy of a tapped node should
collapse onto its supervision gap regardless of q or nominal depth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
w, dpt = SHAPES[-1]
for q, col in zip(Q_GRID, qcols):
    if (w, dpt, q) not in done:
        continue
    acc = done[(w, dpt, q)]["per_out_acc"][-1]
    gaps = supervision_gaps(circuit, taps[q])
    tw = np.flatnonzero(taps[q])
    jitter = (np.random.default_rng(0).random(len(tw)) - 0.5) * 0.25
    axes[0].scatter(gaps[tw] + jitter, acc[tw], s=14, alpha=0.55,
                    color=col, label=f"q={q}")
    axes[1].scatter(node_depths[tw] + jitter, acc[tw], s=14, alpha=0.55,
                    color=col)
for ax, lab in zip(axes, ["supervision gap", "tap depth"]):
    ax.axhline(0.5, color="gray", ls=":", lw=1)
    ax.set(xlabel=lab, ylabel="final eval accuracy", ylim=(0.42, 1.03))
axes[0].legend(fontsize=8)
axes[0].set_title(f"w{w}d{dpt}: acc vs gap (prediction: clean collapse)")
axes[1].set_title("acc vs depth (prediction: mixed at low q)")
plt.tight_layout()

## Gap distributions (the naturally generated difficulty spectrum)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for q, col in zip(Q_GRID, qcols):
    gaps = supervision_gaps(circuit, taps[q])[taps[q]]
    vals, counts = np.unique(gaps, return_counts=True)
    ax.plot(vals, counts / counts.sum(), "o-", color=col, label=f"q={q}")
ax.set(yscale="log", xlabel="supervision gap of tapped nodes",
       ylabel="fraction", title="percolation-generated difficulty distribution")
ax.legend(fontsize=8)
plt.tight_layout()

for (w, dpt, q), d in sorted(done.items()):
    tw = np.flatnonzero(taps[q])
    acc = d["per_out_acc"][-1][tw]
    print(f"w{w}d{dpt} q={q:<5}: tapped acc {acc.mean():.4f}, "
          f"below 0.95: {(acc < 0.95).sum()}/{len(tw)}")

## Notes

- Runs are tagged `_tree3_c2187x7` plus a tap-set hash, so arms never collide
  with each other or with brickwork tasks.
- LR is reused from the hard-task tune at matching shapes; retune if the q=1
  arm looks unstable (its effective loss scale differs most).
- Extending an arm: bump `STEPS` — the resumed run keeps its tap set via the
  config, and longer horizons mainly matter for the low-q deep taps.
- Balanced 3-bit gates are pure parities ~3% of the time, so a few gap-1
  nodes may still straggle; the gap plot will show them as outliers at gap 1.